# The header and the printed marks

Session 3, *Linking and editing*: block 2.

In block 1 you linked every bar of Grieg's *Vöglein* to its place in the print. Now you work on the encoding itself:

1. you read its *header*, the part of the MEI file that says where the encoding comes from, and add what you know about our print and about your own work;
2. you restore the printed marks of one system that the encoding leaves out: dynamics, slurs and pedal marks, with the score and the print side by side;
3. another pair checks your marks against the print, and you check theirs.

This notebook continues in the folder `grieg_edition_work` of `03_grieg_linking.ipynb` and `03_grieg_repository.ipynb`: run those first. You edit the same file, `grieg_edition_work/edition/mei/work.mei`, and after each task you record your change as a commit in your repository.

**How to use this notebook:** run the cells from top to bottom with Shift+Enter. If something goes wrong, the message appears under the cell.

## Setup

The first cell makes sure the packages are installed. The second defines helper functions; it is collapsed because you do not need to read it. The third finds your working file and the file as it was before editing, which the notebook compares it with.

In [ ]:
%pip install -q "camat[validation,notebooks]==0.2.8" pypdf==6.10.0 verovio==6.3.0

In [ ]:
# Helper functions for this notebook. Run this cell; you do not need to read it.
# They follow the same rules as the demos on the course page.

"""Grieg's Vöglein, from the PDF to a linked MEI file: source, pages, boxes, links, check."""

import hashlib
import io
import json
import os
import subprocess
import sys
import tempfile
import time
import urllib.request
import zipfile
from pathlib import Path
import xml.etree.ElementTree as ET

from IPython.display import HTML, Image, display

MEI_NS = "http://www.music-encoding.org/ns/mei"
XML_ID = "{http://www.w3.org/XML/1998/namespace}id"
INPUTS_ZIP = "grieg-edition-inputs.zip"
SYSTEM_COLOURS = [(20, 90, 210), (170, 70, 10), (0, 125, 85), (140, 40, 150), (20, 115, 155)]


def _find(name, start=None):
    start = Path(start or Path.cwd()).resolve()
    for folder in (start, *start.parents):
        for candidate in (folder / name, folder / "notebooks" / name,
                          folder / "sessions/03-linking-and-editing/notebooks" / name):
            if candidate.is_file():
                return candidate
    return None


def sha256_of(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def _keep_or_write(path, data):
    """Write a file once; never replace a file that differs."""
    path = Path(path)
    if path.exists():
        if path.read_bytes() != data:
            raise FileExistsError(f"{path} is already there and differs from the course file. "
                                  "Move it away, or choose a new PROJECT folder, and run the cell again.")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("xb") as output:
        output.write(data)


def _bars(labels):
    labels = list(labels)
    return f"bars {labels[0]}–{labels[-1]}" if len(labels) > 1 else f"bar {labels[0]}"


def _short(path, project):
    try:
        return Path(path).resolve().relative_to(Path(project).resolve().parent)
    except ValueError:
        return Path(path)


# --- 1. The inputs, the working file and the volume ---------------------------

def load_inputs(project):
    """Unpack the course files once and make the working file work.mei."""
    project = Path(project).resolve()
    bundle = _find(INPUTS_ZIP)
    if bundle is None:
        raise FileNotFoundError(f"Put {INPUTS_ZIP} in the same folder as this notebook "
                                "(get_notebooks.py does this), then run this cell again.")
    with zipfile.ZipFile(bundle) as archive:
        expected = json.loads(archive.read("manifest.json"))["files_sha256"]
        files = {name: archive.read(name) for name in expected}
    for name, data in files.items():
        if hashlib.sha256(data).hexdigest() != expected[name]:
            raise ValueError(f"{name} in {bundle.name} is damaged. Download the ZIP again with get_notebooks.py.")
    settings = json.loads(files["settings.json"])
    inputs = project / "inputs"
    for name, data in files.items():
        if name != "manifest.json":
            _keep_or_write(inputs / name, data)
    work = project / "edition/mei/work.mei"
    if not work.exists():
        _keep_or_write(work, files[settings["mei"]])
        made = "Made your working file, a copy of the encoding:"
    else:
        made = "Your working file is already there. It is kept as it is:"
    context = {"project": project, "inputs": inputs, "settings": settings,
               "sources": json.loads(files["sources.json"]), "mei": inputs / settings["mei"], "work": work,
               "saved_boxes": {name.split("/")[1][:-5]: data for name, data in files.items() if name.startswith("boxes/")},
               "volume": project / "sources" / settings["volume_filename"],
               "excerpt": project / "sources" / settings["excerpt_filename"],
               "pages_json": project / "edition/pages/pages.json",
               "alignment": project / "edition/alignment.json"}
    print("The encoding, unchanged:", _short(context["mei"], project))
    print(made, _short(work, project))
    return context


def get_volume(context, allow_download=True):
    """Download the Peters volume from IMSLP once, and check that it is the course's file."""
    witness, volume = context["sources"]["printed_witness"], context["volume"]
    if volume.exists():
        if sha256_of(volume) != witness["sha256"]:
            raise ValueError(f"{volume.name} is not the scan the course uses. Move it away and run the cell again.")
        print("The volume is already there:", _short(volume, context["project"]))
    else:
        if not allow_download:
            raise FileNotFoundError(f"Copy the PDF here: {volume}")
        request = urllib.request.Request(witness["download_url"], headers={"User-Agent": "camat-grieg-seminar"})
        try:
            with urllib.request.urlopen(request, timeout=90) as response:
                data = response.read(witness["bytes"] + 1)
        except OSError as problem:
            raise RuntimeError(f"Could not download the volume from IMSLP ({problem}). Check the internet connection "
                               f"and run the cell again, or copy the PDF from the USB stick to: {volume}") from None
        if len(data) != witness["bytes"] or hashlib.sha256(data).hexdigest() != witness["sha256"]:
            raise ValueError("The download is not the scan the course uses. Run the cell again later.")
        _keep_or_write(volume, data)
        print("Downloaded:", _short(volume, context["project"]))
    print(f"  {witness['bytes'] / 1e6:.1f} MB, {witness['page_count']} pages; "
          f"fingerprint (SHA-256) {witness['sha256'][:12]}…, the same as ours ✓")


# --- 2. Our two pages ----------------------------------------------------------

def cut_pages(context):
    """Copy PDF pages 9 and 10 of the volume into a PDF of their own; the volume stays unchanged."""
    import pypdf
    settings, record = context["settings"], context["sources"]["classroom_pdf"]
    if pypdf.__version__ != settings["pypdf_version"]:
        raise RuntimeError(f"This notebook needs pypdf {settings['pypdf_version']}, but {pypdf.__version__} is installed. "
                           "Run the first cell, restart the kernel, and run the notebook again.")
    volume, excerpt = context["volume"], context["excerpt"]
    before = sha256_of(volume)
    reader = pypdf.PdfReader(volume)
    writer, memory = pypdf.PdfWriter(), io.BytesIO()
    for number in settings["volume_pdf_pages"]:
        writer.add_page(reader.pages[number - 1])
    writer.add_metadata(record["pdf_metadata"])
    writer.write(memory)
    data = memory.getvalue()
    if hashlib.sha256(data).hexdigest() != record["sha256"]:
        raise ValueError("The two-page PDF differs from ours. Ask your teacher.")
    _keep_or_write(excerpt, data)
    if sha256_of(volume) != before or len(pypdf.PdfReader(excerpt).pages) != 2:
        raise ValueError("Something went wrong while copying the pages. Ask your teacher.")
    print("Our two pages:", _short(excerpt, context["project"]))
    print(f"\n{'In the volume':<16}{'In our PDF':<13}Printed page")
    for mapping in record["page_mapping"]:
        print(f"PDF page {mapping['original_pdf_page']:<7}PDF page {mapping['excerpt_pdf_page']:<4}{mapping['printed_page']}")
    print("\nThe volume is unchanged ✓")


# --- 3. The pages as images ------------------------------------------------------

def draw_pages(context):
    """Draw both pages as PNG images at 300 dpi; CAMAT writes pages.json beside them."""
    from camat.pdf_facsimile import check_poppler, render_pdf_pages
    check_poppler()
    manifest = render_pdf_pages(context["excerpt"], context["pages_json"].parent, dpi=context["settings"]["dpi"])
    if manifest["pdf_sha256"] != context["sources"]["classroom_pdf"]["sha256"]:
        raise ValueError("The images do not come from our two-page PDF. Run the cells of parts 1 and 2 again.")
    for page in manifest["pages"]:
        if [page["width"], page["height"]] != context["settings"]["image_pixels"]:
            raise ValueError(f"{page['image']} is {page['width']} × {page['height']} pixels; we expected "
                             f"{' × '.join(map(str, context['settings']['image_pixels']))}. Ask your teacher.")
        print(f"PDF page {page['pdf_page_number']}: {_short(context['pages_json'].parent / page['image'], context['project'])}, "
              f"{page['width']} × {page['height']} pixels")
    print("CAMAT's record of the images:", _short(context["pages_json"], context["project"]))
    (context["pages_json"].parent / "boxes").mkdir(exist_ok=True)


def _manifest(context):
    if not context["pages_json"].is_file():
        raise FileNotFoundError("The page images are not there yet: run the cell of part 3 first.")
    return json.loads(context["pages_json"].read_text())


def _page(context, number):
    return next(page for page in _manifest(context)["pages"] if page["pdf_page_number"] == number)


def _picture(context, page, width):
    from PIL import Image as PILImage
    with PILImage.open(context["pages_json"].parent / page["image"]) as source:
        picture = source.convert("RGB")
    picture.thumbnail((width, width * 2))
    return picture


def _show_picture(picture):
    buffer = io.BytesIO()
    picture.save(buffer, format="PNG")
    display(Image(data=buffer.getvalue()))


def show_pages(context, width=560):
    """Both pages side by side, small."""
    from PIL import Image as PILImage
    pictures = [_picture(context, page, width) for page in _manifest(context)["pages"]]
    sheet = PILImage.new("RGB", (sum(p.width for p in pictures) + 20, max(p.height for p in pictures)), "white")
    sheet.paste(pictures[0], (0, 0))
    sheet.paste(pictures[1], (pictures[0].width + 20, 0))
    _show_picture(sheet)


# --- 4 and 5. Describe a page -----------------------------------------------------

def _systems(page):
    return [s["measures"] if isinstance(s, dict) else s for s in page.get("systems", [])]


def _sequence(layout):
    sequence = []
    for number in sorted(layout):
        for system in layout[number].get("systems", []):
            sequence.extend(system[1:] if sequence and system[0] == sequence[-1] else system)
    return sequence


def describe_page(context, line, number, save=True):
    """Check one page's line against the MEI file and save it in pages.json.

    CAMAT checks a whole movement. While the other page is not described yet, a page
    passes if its bars are an unbroken run of the MEI's bars, in order.
    """
    from camat.pdf_facsimile import apply_page_layout, check_page_layout, parse_page_layout
    ours = context["settings"]["layout"][number - 1]
    if not (line or "").strip():
        line = ours
        print(f"You have not written the line yet, so the notebook uses ours:\n    {ours}\n")
    try:
        layout = parse_page_layout(line, page_count=2)
    except ValueError as problem:
        print(f"CAMAT cannot read the line: {problem}\nNothing was saved.")
        return
    if set(layout) != {number}:
        print(f"This line describes PDF page {', '.join(map(str, layout))}; this cell is for PDF page {number}. "
              f"Start the line with {number} |. Nothing was saved.")
        return
    # The other page, as far as it is described already.
    for page in _manifest(context)["pages"]:
        if page["pdf_page_number"] != number and page["role"] == "music":
            layout[page["pdf_page_number"]] = {"role": "music", "printed_page": page.get("printed_page"),
                                               "systems": _systems(page)}
    rest = []
    try:
        check_page_layout(layout, context["work"])
    except ValueError as problem:
        details = getattr(problem, "details", {})
        bars, sequence = details.get("bars", []), _sequence(layout)
        starts = [i for i, bar in enumerate(bars) if sequence and bar == sequence[0]]
        unbroken = bool(starts) and bars[starts[0]:starts[0] + len(sequence)] == sequence
        if len(layout) == 2 or getattr(problem, "code", None) != "bars" or details.get("extra") or not unbroken:
            print(f"{problem}\nNothing was saved. Compare the line with the page, correct it and run the cell again.")
            return
        rest = details["missing"]
    if not save:
        print("CAMAT accepts this line (not saved).")
        return
    apply_page_layout(context["pages_json"], layout)
    page = layout[number]
    print(f"✓ Saved: PDF page {number} is printed page {page['printed_page']}, with {len(page['systems'])} systems.")
    for index, labels in enumerate(page["systems"], 1):
        print(f"    system {index}: {_bars(labels)} ({len(labels)} bars)")
    if rest:
        print(f"CAMAT checked the bars against the MEI file: none missing in between, in the right order.\n"
              f"Not described yet: {_bars(rest)}, on the other page.")
    else:
        print("CAMAT checked both pages against the MEI file: all its bars, each once, in the right order.")


# --- Boxes ---------------------------------------------------------------------

def box_file(context, number):
    return context["pages_json"].parent / "boxes" / f"{_page(context, number)['page_id']}.json"


def box_prompt(context, number):
    """A prompt for a chat model that can read images."""
    page = _page(context, number)
    counts = ", ".join(str(len(s)) for s in _systems(page)) or "?"
    image = context["pages_json"].parent / page["image"]
    print("1. In the chat, attach this image:", _short(image, context["project"]))
    print("2. Copy the prompt between the lines into the chat:\n")
    print("-" * 70)
    print(f"""The attached image is a page of printed piano music, {page['width']} pixels wide and
{page['height']} pixels high. It has {len(_systems(page)) or 'several'} systems of two staves each; from top to
bottom they have {counts} bars. Find every bar and give it a box from the barline
where it begins (or the start of the system) to the barline where it ends,
covering both staves. A short bar before or after a repeat sign is a bar too.

Answer only with JSON in this form, nothing else. Coordinates are pixels of the
full image: x from the left edge, y from the top edge. ulx, uly is the upper
left corner of a box, lrx, lry the lower right corner.

{{"measures": [{{"ulx": 100, "uly": 200, "lrx": 300, "lry": 400}}, ...]}}""")
    print("-" * 70)
    print(f"\n3. Save the answer (only the JSON) as: {_short(box_file(context, number), context['project'])}")


def _box_source(context, path, page):
    """Where a box file comes from. The detector's run record comes first: for the same
    pixels the detector can answer with exactly the bytes of the saved file."""
    data = path.read_bytes()
    run = path.with_suffix(".run.json")
    if run.is_file():
        edited = json.loads(run.read_text()).get("response_sha256") != hashlib.sha256(data).hexdigest()
        return "the detector's answer" + (", edited since" if edited else "")
    if data == context["saved_boxes"].get(page["page_id"]):
        return "the boxes we saved for the course"
    return "your own file (for example the chat's answer)"


def find_boxes(context, number, send=False):
    """Use the box file of a page: the chat's answer, the detector's answer, or the saved boxes."""
    from camat.pdf_facsimile import detect_measure_boxes
    page = _page(context, number)
    if page["role"] != "music":
        print(f"Describe PDF page {number} first (the cell above), then run this cell again.")
        return
    path = box_file(context, number)
    saved = context["saved_boxes"][page["page_id"]]
    if send:
        if path.exists() and _box_source(context, path, page) != "the boxes we saved for the course":
            print(f"There is already a box file for this page, {_box_source(context, path, page)}:\n"
                  f"    {_short(path, context['project'])}\nIt is kept. To send the page again, delete or rename "
                  "that file, then run this cell again.")
        else:
            # The detector gets every described page without a box file, so check the other page first.
            others = [p for p in _manifest(context)["pages"] if p["role"] == "music" and p["page_id"] != page["page_id"]
                      and not (path.parent / f"{p['page_id']}.json").exists()]
            if others:
                print(f"PDF page {others[0]['pdf_page_number']} is described but has no boxes yet: "
                      "run its box cell first. Nothing was sent.")
                return
            path.unlink(missing_ok=True)  # the saved boxes give way to the detector's answer
            print(f"Sending PDF page {number} to the Edirom measure detector (measure-detector.edirom.de) …")
            try:
                detect_measure_boxes(context["pages_json"], allow_upload=True)
            except Exception as problem:  # network, server or answer
                print(f"The detector did not answer ({problem}).\nRun the cell again in a minute, or use the "
                      "saved boxes: set SEND_TO_DETECTOR = False and run the cell again.")
                return
    elif not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(saved)
        print("There is no box file for this page yet, so the notebook uses the boxes we saved for the course.\n"
              "To find them yourself, use the chat or the detector (see above) and run this cell again.\n")
    print(f"Boxes for PDF page {number}: {_box_source(context, path, page)}")
    print(f"    {_short(path, context['project'])}")
    _report(context, page, path)


def _read_boxes(page, path):
    """(systems, problem): CAMAT's grouping of the boxes into systems, or what is wrong with the file."""
    from camat.pdf_facsimile import group_measure_boxes_into_systems
    if not path.is_file():
        return None, f"There is no box file yet: {path.name}"
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as problem:
        return None, (f"{path.name} cannot be read as JSON: line {problem.lineno}, column {problem.colno}: {problem.msg}. "
                      "Often a comma or a bracket is missing, or text from the chat came along. Fix it and save.")
    try:
        return group_measure_boxes_into_systems(payload, page), None
    except (ValueError, KeyError, TypeError) as problem:
        return None, f"{path.name}: {problem}. Fix it and save."


def _report(context, page, path):
    systems, problem = _read_boxes(page, path)
    if problem:
        print("✗", problem)
        return
    needed = _systems(page)
    print(f"    {sum(map(len, systems))} boxes in {len(systems)} rows; the page has {len(needed)} systems "
          f"with {sum(map(len, needed))} bars.")


# --- The live box overlay -----------------------------------------------------------

_watchers = {}


def _overlay(context, number, base):
    """(status as HTML, PNG bytes): the page with the boxes of its box file, numbered with their bars."""
    from PIL import ImageDraw, ImageFont
    page = _page(context, number)
    path = box_file(context, number)
    scale = base.width / page["width"]
    needed = _systems(page)
    systems, problem = _read_boxes(page, path)
    picture = base.copy()
    lines = []
    if problem:
        lines.append(f"<b style='color:#b91c1c'>✗ {problem}</b>")
    else:
        draw = ImageDraw.Draw(picture, "RGBA")
        try:
            font = ImageFont.truetype("DejaVuSans-Bold.ttf", 22)
        except OSError:
            font = ImageFont.load_default(22)
        for row, boxes in enumerate(systems):
            colour = SYSTEM_COLOURS[row % len(SYSTEM_COLOURS)]
            labels = needed[row] if row < len(needed) and len(needed[row]) == len(boxes) else None
            for index, box in enumerate(boxes):
                ulx, uly, lrx, lry = (round(v * scale) for v in box["bbox"])
                draw.rectangle([ulx, uly, lrx, lry], fill=(*colour, 22), outline=(*colour, 235), width=3)
                label = labels[index] if labels else f"{row + 1}.{index + 1}?"
                corner = (ulx + 6, lry - 6)
                left, top, right, bottom = draw.textbbox(corner, label, font=font, anchor="lb")
                draw.rectangle([left - 3, top - 2, right + 3, bottom + 2], fill=(255, 255, 255, 225))
                draw.text(corner, label, font=font, fill=(*colour, 255), anchor="lb")
        for row in range(max(len(systems), len(needed))):
            have = len(systems[row]) if row < len(systems) else 0
            if row < len(needed):
                fits = have == len(needed[row])
                lines.append(f"<span style='color:{'#047857' if fits else '#b91c1c'}'>{'✓' if fits else '✗'} "
                             f"system {row + 1}: {have} boxes for {_bars(needed[row])} ({len(needed[row])} bars)</span>")
            else:
                lines.append(f"<span style='color:#b91c1c'>✗ row {row + 1}: {have} boxes, but the page has "
                             f"only {len(needed)} systems</span>")
        if [len(s) for s in systems] == [len(s) for s in needed]:
            lines.append("<b>The counts fit. Now look: does each number sit on its bar, in both staves?</b>")
        else:
            lines.append("<b>A label like 2.3? is the 3rd box of a row whose count does not fit.</b>")
    lines.append(f"<span style='color:#6b7280'>{path.name} · drawn at {time.strftime('%H:%M:%S')}</span>")
    buffer = io.BytesIO()
    picture.save(buffer, format="PNG")
    return "<br>".join(lines), buffer.getvalue()


class BoxOverlay:
    """The page with its numbered boxes, drawn again each time the box file or pages.json is saved."""

    def __init__(self, context, number, width=900, interval=1.0):
        import ipywidgets as widgets
        self.context, self.number = context, number
        self.path = box_file(context, number)
        self.base = _picture(context, _page(context, number), width)
        self.status = widgets.HTML()
        self.image = widgets.Image(format="png", layout=widgets.Layout(width=f"{width}px", max_width="100%"))
        self.box = widgets.VBox([self.status, self.image])
        self.interval, self.callback, self.seen = interval, None, None

    def _stamp(self):
        stamps = []
        for path in (self.path, self.context["pages_json"]):
            try:
                stat = path.stat()
                stamps.append((stat.st_mtime_ns, stat.st_size))
            except OSError:
                stamps.append(None)
        return tuple(stamps)

    def draw(self):
        self.seen = self._stamp()
        self.status.value, self.image.value = _overlay(self.context, self.number, self.base)

    def _poll(self):
        if self._stamp() != self.seen:
            try:
                self.draw()
            except Exception as problem:  # keep watching while the file is half written
                self.status.value = f"<b style='color:#b91c1c'>Could not draw: {problem}</b>"

    def start(self):
        from tornado.ioloop import PeriodicCallback
        self.draw()
        self.callback = PeriodicCallback(self._poll, int(self.interval * 1000))
        self.callback.start()
        return self

    def stop(self):
        if self.callback is not None:
            self.callback.stop()
            self.callback = None
            self.status.value += "<br><i>Stopped watching the file.</i>"


def watch_boxes(context, number):
    """Show the live box overlay for one page; it redraws when the box file is saved."""
    page = _page(context, number)
    if page["role"] != "music":
        print(f"Describe PDF page {number} first, then run this cell again.")
        return None
    old = _watchers.pop(page["page_id"], None)
    if old is not None:
        old.stop()
    try:
        overlay = BoxOverlay(context, number).start()
    except ImportError:
        print("The live picture needs ipywidgets, so this picture does not follow the file. "
              "Run this cell again after each change. The boxes as they are now:")
        status, png = _overlay(context, number, _picture(context, page, 900))
        display(HTML(status))
        display(Image(data=png))
        return None
    _watchers[page["page_id"]] = overlay
    display(overlay.box)
    return overlay


def stop_watching():
    for overlay in list(_watchers.values()):
        overlay.stop()
    _watchers.clear()


def list_boxes(context, number):
    """Each bar with the position of its box in the file and its four numbers."""
    page = _page(context, number)
    path = box_file(context, number)
    systems, problem = _read_boxes(page, path)
    if problem:
        print("✗", problem)
        return
    needed = _systems(page)
    print(f"{path.name}: bar, box number in the file (counted from 1), and its four numbers (left, top, right, bottom)")
    for row, boxes in enumerate(systems):
        labels = needed[row] if row < len(needed) and len(needed[row]) == len(boxes) else None
        print(f"System {row + 1}")
        for index, box in enumerate(boxes):
            bar = f"bar {labels[index]}" if labels else f"{row + 1}.{index + 1}?"
            print(f"    {bar:<9} box {box['box_index'] + 1:>2}: {tuple(box['bbox'])}")


# --- 6. Align, review, link ------------------------------------------------------------

def align_pages(context, width=560):
    """Pair the boxes with the bars, system by system, and draw CAMAT's overlays of both pages."""
    from camat.pdf_facsimile import build_measure_alignment, check_page_layout, draw_alignment_overlays
    from PIL import Image as PILImage
    manifest = _manifest(context)
    layout = {p["pdf_page_number"]: {"role": "music", "systems": _systems(p)}
              for p in manifest["pages"] if p["role"] == "music"}
    if len(layout) != 2:
        print("Describe both pages first (parts 4 and 5), then run this cell again.")
        return False
    try:
        check_page_layout(layout, context["work"])
    except ValueError as problem:
        print(f"The two page lines do not fit the MEI file:\n{problem}\nCorrect the line in part 4 or 5 and run it again.")
        return False
    missing = [p["pdf_page_number"] for p in manifest["pages"] if not box_file(context, p["pdf_page_number"]).exists()]
    if missing:
        print(f"PDF page {missing[0]} has no box file yet: run its box cell (part 4 or 5), then this cell.")
        return False
    try:
        alignment = build_measure_alignment(context["pages_json"], output=context["alignment"])
    except ValueError as problem:
        context["alignment"].unlink(missing_ok=True)
        print("CAMAT did not pair the boxes with the bars: in some systems the number of boxes and bars differs.\n"
              f"{problem}\n(actual = boxes found, expected = bars in the description.) Fix the boxes, "
              "check them in the live picture, and run this cell again.")
        return False
    data = json.loads(alignment.read_text())
    print(f"✓ CAMAT paired {len(data['regions'])} boxes with {len(data['regions'])} bars on {len(data['pages'])} pages.")
    outputs = draw_alignment_overlays(alignment, context["project"] / "edition/overlays")
    pictures = []
    for output in outputs:
        with PILImage.open(output) as picture:
            picture = picture.convert("RGB")
            picture.thumbnail((width, width * 2))
            pictures.append(picture)
    sheet = PILImage.new("RGB", (sum(p.width for p in pictures) + 20, max(p.height for p in pictures)), "white")
    sheet.paste(pictures[0], (0, 0))
    sheet.paste(pictures[1], (pictures[0].width + 20, 0))
    _show_picture(sheet)
    print("Full size:", ", ".join(str(_short(output, context["project"])) for output in outputs))
    return True


def record_review(context, names):
    from camat.pdf_facsimile import approve_alignment_review
    if not context["alignment"].is_file():
        print("Run the cell above first, until it shows ✓.")
        return
    if not (names or "").strip():
        print("Write your names after REVIEWED_BY = and run the cell again. Without them the links can still be made, "
              "but CAMAT records both pages as unreviewed.")
        return
    approve_alignment_review(context["alignment"], names)
    review = json.loads(context["alignment"].read_text())["review"]
    for page_id, entry in review.items():
        print(f"✓ {page_id}: reviewed by {entry['reviewer']} on {entry['date']}")


def _music(root):
    """The music of a file without the links: to check that linking changed nothing else."""
    ns = f"{{{MEI_NS}}}"
    body = root.find(f"{ns}music/{ns}body")

    def walk(element):
        if element.tag in {ns + "pb", ns + "sb"}:
            return None
        attributes = {k: v for k, v in element.attrib.items() if k != "facs"}
        if element.tag == ns + "mdiv":
            attributes.pop("n", None)  # linking numbers the one movement n="1"
        return (element.tag, sorted(attributes.items()), (element.text or "").strip(),
                [value for child in element if (value := walk(child)) is not None])
    return walk(body)


def _linked_text(context, text):
    """(text, report, breaks): an MEI text with this edition's links, made beside work.mei
    so that the path to the page images is right for files in edition/mei and edition/checkpoints."""
    from camat.pdf_facsimile import integrate_pdf_facsimile, number_single_mdiv, write_page_breaks
    handle = tempfile.NamedTemporaryFile(prefix=".linking-", suffix=".mei", dir=context["work"].parent, delete=False)
    handle.close()
    candidate = Path(handle.name)
    try:
        candidate.write_text(number_single_mdiv(text), encoding="utf-8")
        report = integrate_pdf_facsimile(candidate, context["pages_json"], context["alignment"], candidate)
        breaks = write_page_breaks(candidate, context["pages_json"], candidate)
        return candidate.read_text(encoding="utf-8"), report, breaks
    finally:
        candidate.unlink(missing_ok=True)


def _replace(path, text):
    path.parent.mkdir(parents=True, exist_ok=True)
    handle = tempfile.NamedTemporaryFile(prefix=".saving-", suffix=".mei", dir=path.parent, delete=False)
    handle.close()
    Path(handle.name).write_text(text, encoding="utf-8")
    os.replace(handle.name, path)


def link_work(context):
    """Add the links to work.mei, keeping a copy of it as it was before.

    The unchanged encoding gets the same links, as edition/checkpoints/before-editing.mei:
    block 2 compares work.mei with it, so that its list of changes shows only your edits.
    """
    project, work = context["project"], context["work"]
    if not context["alignment"].is_file():
        print("Run the cell that pairs the boxes with the bars first, until it shows ✓.")
        return
    try:
        root = ET.parse(work).getroot()
    except ET.ParseError as error:
        line, column = error.position
        print(f"{work.name} cannot be read as XML: the problem is at line {line}, column {column}.\n"
              "Fix it in the editor first (Ctrl+Z, on a Mac Cmd+Z, undoes your last change), save, "
              "and run this cell again. Nothing was changed.")
        return
    text = work.read_text(encoding="utf-8")
    try:
        linked, report, breaks = _linked_text(context, text)
        baseline, _, _ = _linked_text(context, context["mei"].read_text(encoding="utf-8"))
    except ValueError as problem:
        print(f"The links were not added: {problem}\nNothing was changed.")
        return
    if _music(ET.fromstring(linked.encode("utf-8"))) != _music(root):
        print(f"Linking would change more than the links in {work.name}; nothing was saved. Ask your teacher.")
        return
    backup = project / "edition/checkpoints/work-before-linking.mei"
    if not backup.exists():
        _keep_or_write(backup, text.encode("utf-8"))
    _replace(work, linked)
    _replace(project / "edition/checkpoints/before-editing.mei", baseline)
    if "<facsimile" in text:
        print(f"✓ Updated the links in your working file, for the boxes as they are now: {_short(work, project)}")
    else:
        print(f"✓ Added the links to your working file: {_short(work, project)}")
    print(f"  {report['measures_linked']} bars point to their boxes on {report['surfaces']} pages; "
          f"{breaks['pb']} page breaks and {breaks['sb']} system breaks follow the print.")
    if report.get("unreviewed_pages"):
        print(f"  Not reviewed yet: {', '.join(report['unreviewed_pages'])} (write your names in the cell above).")
    print("  The music is unchanged: only the links are new.")
    print(f"  The file as it was before the links: {_short(backup, project)}")


# --- 7. Follow one link ---------------------------------------------------------------------

def _line_of(lines, text, start=0):
    for number in range(start, len(lines)):
        if text in lines[number]:
            return number
    return None


def follow_link(context, bar, width=900):
    """The chain from <measure n="bar"> to its zone, page and image, with the line numbers of work.mei."""
    work = context["work"]
    lines = work.read_text(encoding="utf-8").split("\n")
    root = ET.parse(work).getroot()
    music = root.find(f"{{{MEI_NS}}}music")
    measure = next((m for m in music.iter(f"{{{MEI_NS}}}measure") if m.get("n") == str(bar)), None)
    if measure is None:
        print(f"There is no <measure n=\"{bar}\"> in the music. Choose a bar from 1 to 39.")
        return
    if not measure.get("facs"):
        print(f"Bar {bar} has no link yet (no facs). Run the cell of part 6 first.")
        return
    zone_id = measure.get("facs").lstrip("#")
    zone = surface = None
    for candidate_surface in music.iter(f"{{{MEI_NS}}}surface"):
        for candidate_zone in candidate_surface.iter(f"{{{MEI_NS}}}zone"):
            if candidate_zone.get(XML_ID) == zone_id:
                zone, surface = candidate_zone, candidate_surface
    graphic = surface.find(f"{{{MEI_NS}}}graphic")
    start = _line_of(lines, "<music")
    steps = [("the bar", _line_of(lines, f'xml:id="{measure.get(XML_ID)}"', start), "facs points to the zone"),
             ("the zone", _line_of(lines, f'xml:id="{zone_id}"', start), "the box: ulx, uly (upper left), lrx, lry (lower right)"),
             ("its page", _line_of(lines, f'xml:id="{surface.get(XML_ID)}"', start), "the <surface> holding the zone"),
             ("the image", _line_of(lines, f'target="{graphic.get("target")}"', start), "the image file, seen from work.mei")]
    for name, number, note in steps:
        print(f"{name:<10} line {number + 1:>4}: {lines[number].strip()}")
        print(f"{'':<21}↑ {note}")
    box = [int(float(zone.get(k))) for k in ("ulx", "uly", "lrx", "lry")]
    image = (work.parent / graphic.get("target")).resolve()
    from PIL import Image as PILImage, ImageDraw
    with PILImage.open(image) as source:
        picture = source.convert("RGB")
    margin = 120
    region = (max(0, box[0] - margin), max(0, box[1] - margin),
              min(picture.width, box[2] + margin), min(picture.height, box[3] + margin))
    draw = ImageDraw.Draw(picture, "RGBA")
    draw.rectangle(box, fill=(200, 30, 30, 25), outline=(200, 30, 30, 240), width=8)
    crop = picture.crop(region)
    crop.thumbnail((width, width))
    print(f"\nThe box ({', '.join(map(str, box))}) on {image.name}:")
    _show_picture(crop)


# --- 8. The check -----------------------------------------------------------------------

def _validate(path, folder):
    subprocess.run([sys.executable, "-m", "camat.mei_validation", str(path), "--output-dir", str(folder), "--verovio"],
                   text=True, capture_output=True, timeout=300)
    findings = Path(folder) / "findings.json"
    return json.loads(findings.read_text()) if findings.is_file() else None


def _finding_key(finding):
    # Line numbers move when lines are added, and Verovio names elements without an xml:id
    # with new random IDs on each run, so neither is compared.
    message = finding.get("message", "")
    if finding.get("check") == "verovio_log" and " with @xml:id " in message:
        message = message.split(" with @xml:id ")[0] + " " + message.split(" has ", 1)[-1]
    return tuple(finding.get(k) for k in ("severity", "check", "element", "xml_id", "measure_n", "staff_n", "layer_n")) + (message,)


def check_links(context):
    """Check work.mei with CAMAT's MEI checker; show only what is new since before the links."""
    project, work = context["project"], context["work"]
    if "<facsimile" not in work.read_text(encoding="utf-8"):
        print(f"{work.name} has no links yet. Run the cell of part 6 until it shows ✓, then this cell.")
        return
    backup = project / "edition/checkpoints/work-before-linking.mei"
    now = _validate(work, project / "edition/validation-linked")
    then = _validate(backup, project / "edition/validation-before-linking")
    if now is None or then is None:
        print("The checker did not run. Run the cell again; if it still fails, ask your teacher.")
        return
    known = {}
    for finding in then:
        known[_finding_key(finding)] = known.get(_finding_key(finding), 0) + 1
    new = []
    for finding in now:
        key = _finding_key(finding)
        if known.get(key):
            known[key] -= 1
        else:
            new.append(finding)
    print(f"Checked {work.name} against the MEI rules (schema), its links and the page images, and with Verovio.")
    if not new:
        print("✓ No new problems: every bar points to a zone in the file, and the file finds both page images.")
        print(f"  (The encoding already had {len(then)} remarks before the links; they are not shown here.)")
        return
    for finding in new:
        where = f" (bar {finding['measure_n']})" if finding.get("measure_n") else ""
        mark = "✗" if finding["severity"] == "error" else "!"
        print(f"  {mark} line {finding.get('line') or '?'}{where}: {finding['message']}")
    if any(finding["severity"] == "error" for finding in new):
        print("Fix the lines marked ✗, save, and run this cell again.")
    print(f"The full report is in {_short(project / 'edition/validation-linked', project)}.")

"""Grieg's Vöglein, block 2: the header and the printed marks, with the live score.

Pasted after grieg_linking.py in the same cell; it uses that file's functions.
"""

import copy
import datetime
import re
import difflib
import textwrap
from fractions import Fraction

MARKS = {"dynam", "tempo", "hairpin", "slur", "pedal", "dir", "fing", "octave"}
HEADER_TASKS = {"identifier", "change", "application"}
ET.register_namespace("", MEI_NS)  # measures shown without a namespace prefix


def _name(element):
    return element.tag.split("}")[-1]


def _text(element):
    """The words of an element and its children, with a space where they meet."""
    if element is None:
        return ""
    return re.sub(r" ([.,;:)])", r"\1", " ".join(" ".join(element.itertext()).split()))


def _pitch(note):
    sign = {"f": "♭", "s": "♯", "n": "", "ff": "𝄫", "ss": "𝄪"}.get(note.get("accid") or note.get("accid.ges") or "n", "")
    return f"{note.get('pname', '?').upper()}{sign}{note.get('oct', '')}"


def _read(path):
    """Parse an MEI file; on an XML error say what to do and return None."""
    try:
        return ET.parse(path).getroot()
    except ET.ParseError as error:
        line, column = error.position
        print(f"{Path(path).name} cannot be read as XML: the problem is at line {line}, column {column}.\n"
              "Usually a tag is not closed, or a < or a quotation mark is missing, at or just before that line.\n"
              "Undo your last change in the editor (Ctrl+Z, on a Mac Cmd+Z), save, and run this cell again.")
        return None


def _nth_line(lines, text, n=1):
    """The line number (from 1) of the n-th line containing text."""
    found = [number for number, line in enumerate(lines, 1) if text in line]
    return found[n - 1] if len(found) >= n else None


# --- The edition and your system -------------------------------------------------

def open_edition(project):
    """Continue from 03_grieg_linking: the linked work.mei and the file before editing."""
    project = Path(project).resolve()
    work = project / "edition/mei/work.mei"
    before = project / "edition/checkpoints/before-editing.mei"
    if not (before.is_file() and work.is_file() and "<facsimile" in work.read_text(encoding="utf-8")):
        raise FileNotFoundError("This notebook continues from 03_grieg_linking.ipynb. Run that notebook to the end "
                                "first, in the same folder, then run this cell again.")
    context = load_inputs(project)
    context["before"] = before
    print("The file before editing (do not edit this one):", _short(before, project))
    return context


def choose_system(context, page, system):
    """Remember which system of which printed page this pair restores."""
    pages = {p.get("printed_page"): p for p in _manifest(context)["pages"]}
    if str(page) not in pages:
        print(f"Page {page} is not one of ours: write 42 or 43.")
        return
    systems = _systems(pages[str(page)])
    if not 1 <= int(system) <= len(systems):
        print(f"Page {page} has systems 1 to {len(systems)}.")
        return
    bars = systems[int(system) - 1]
    context["system"] = {"page": str(page), "number": int(system), "bars": bars}
    print(f"Your system: page {page}, system {system}, {_bars(bars)}.")


def _system(context):
    if "system" not in context:
        raise ValueError("Run the cell with PAGE and SYSTEM in part 2 first.")
    return context["system"]


def _boxes(context):
    """{bar: (image, (left, top, right, bottom))} from the links of the file before editing."""
    music = ET.parse(context["before"]).getroot().find(f"{{{MEI_NS}}}music")
    zones = {}
    for surface in music.iter(f"{{{MEI_NS}}}surface"):
        image = (context["before"].parent / surface.find(f"{{{MEI_NS}}}graphic").get("target")).resolve()
        for zone in surface.iter(f"{{{MEI_NS}}}zone"):
            zones[zone.get(XML_ID)] = (image, tuple(int(float(zone.get(k))) for k in ("ulx", "uly", "lrx", "lry")))
    return {m.get("n"): zones[m.get("facs").lstrip("#")]
            for m in music.iter(f"{{{MEI_NS}}}measure") if (m.get("facs") or "").lstrip("#") in zones}


def _crop(image, box, labels=(), width=1100, margin=40):
    from PIL import Image as PILImage, ImageDraw, ImageFont
    with PILImage.open(image) as source:
        picture = source.convert("RGB")
    draw = ImageDraw.Draw(picture, "RGBA")
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", 44)
    except OSError:
        font = ImageFont.load_default(44)
    for x, y, label in labels:
        left, top, right, bottom = draw.textbbox((x, y), label, font=font, anchor="lt")
        draw.rectangle([left - 6, top - 4, right + 6, bottom + 4], fill=(255, 255, 255, 230))
        draw.text((x, y), label, font=font, fill=(20, 90, 210, 255), anchor="lt")
    region = (max(0, box[0] - margin), max(0, box[1] - margin),
              min(picture.width, box[2] + margin), min(picture.height, box[3] + margin))
    crop = picture.crop(region)
    crop.thumbnail((width, width))
    _show_picture(crop)


def show_system(context):
    """Your system in the print, with its bar numbers."""
    bars = _system(context)["bars"]
    boxes = _boxes(context)
    image = boxes[bars[0]][0]
    union = (min(boxes[b][1][0] for b in bars), min(boxes[b][1][1] for b in bars),
             max(boxes[b][1][2] for b in bars), max(boxes[b][1][3] for b in bars))
    labels = [(boxes[b][1][0] + 30, union[1] + 10, f"bar {b}") for b in bars]
    _crop(image, union, labels)


# --- The header ----------------------------------------------------------------------

def _who(change, ids):
    names = [_text(e) for e in change.iter() if _name(e) in ("persName", "name", "corpName")]
    for pointer in (change.get("resp") or "").split():
        target = ids.get(pointer.lstrip("#"))
        if target is None:
            names.append(f"{pointer} (not in the file!)")
        else:
            names.append(_text(target.find(f"{{{MEI_NS}}}name")) if _name(target) == "application" else _text(target))
    return ", ".join(names) or "?"


def _what(change, ids):
    description = change.find(f"{{{MEI_NS}}}changeDesc")
    text = _text(description)
    for ref in description.iter(f"{{{MEI_NS}}}ref") if description is not None else []:
        target = (ref.get("target") or "").lstrip("#")
        if (ref.get("target") or "").startswith("#") and target not in ids:
            text += f" [the link #{target} points to nothing in the file]"
    return text


def _date(change):
    date = change.find(f"{{{MEI_NS}}}date")
    if date is None:
        return "?"
    return (date.get("isodate") or date.get("notafter") or date.get("notbefore") or _text(date) or "?")[:10]


def _address(identifier):
    """A web address without its http:// part, shortened, so that the output has no live link."""
    text = _text(identifier).split("://", 1)[-1]
    return text if len(text) <= 70 else text[:40] + "…" + text[-28:]


def show_header(context):
    """The header of work.mei in plain words, with the lines where to find it."""
    root = _read(context["work"])
    if root is None:
        return
    lines = context["work"].read_text(encoding="utf-8").split("\n")
    m = f"{{{MEI_NS}}}"
    head = root.find(f"{m}meiHead")
    ids = {e.get(XML_ID): e for e in root.iter() if e.get(XML_ID)}
    title_stmt = head.find(f"{m}fileDesc/{m}titleStmt")
    pub = head.find(f"{m}fileDesc/{m}pubStmt")
    print(f"The header: lines {_nth_line(lines, '<meiHead')}–{_nth_line(lines, '</meiHead')} of {context['work'].name}\n")
    title = title_stmt.find(f"{m}title")
    subtitle = title.find(f"{m}titlePart")
    print("Title:      ", (title.text or "").strip() + (f" ({_text(subtitle)})" if subtitle is not None else ""))
    print("Composer:   ", _text(title_stmt.find(f"{m}composer")))
    print("Encoded by: ", ", ".join(_text(p) for p in title_stmt.iter(f"{m}persName") if p.get("role") == "encoder"))
    print("Published:  ", _text(pub.find(f"{m}publisher")), _text(pub.find(f"{m}date")))
    print("Rights:     ", textwrap.shorten(_text(pub.find(f".//{m}useRestrict")), 90))
    manifestations = head.findall(f"{m}manifestationList/{m}manifestation")
    print(f"\nSources: {len(manifestations)} <manifestation>s")
    for number, source in enumerate(manifestations, 1):
        title = _text(source.find(f"{m}titleStmt/{m}title"))
        people = [f"{_text(p)} ({p.get('role')})" for p in source.find(f"{m}titleStmt").iter()
                  if _name(p) in ("persName", "corpName") and p.get("role") != "composer"]
        people += [_text(p) for p in source.iter(f"{m}corpName") if p not in source.find(f"{m}titleStmt").iter()]
        date = source.find(f".//{m}pubStmt/{m}date")
        plate = source.find(f".//{m}plateNum")
        print(f"  {number}. line {_nth_line(lines, '<manifestation>', number)}: {title}")
        print(f"     {', '.join(people)}" + (f"; {_text(date) or date.get('isodate')}" if date is not None else "")
              + (f"; plate {_text(plate)}" if plate is not None else ""))
        for identifier in source.findall(f"{m}identifier"):
            value = _address(identifier) if identifier.get("type") == "URI" else _text(identifier)
            print(f"     identifier ({identifier.get('type')}): {value}")
    changes = head.findall(f"{m}revisionDesc/{m}change")
    print(f"\nChanges: {len(changes)} <change>s, from line {_nth_line(lines, '<revisionDesc')}")
    for change in changes:
        what = textwrap.wrap(_what(change, ids), 64) or ["?"]
        print(f"  {change.get('n') or '?':>3}  {_date(change):<10}  {textwrap.shorten(_who(change, ids), 26):<26}  {what[0]}")
        for more in what[1:]:
            print(f"{'':45}{more}")


def header_lines(context, names):
    """The lines to add to the header, ready to paste, with where they go."""
    root = _read(context["work"])
    if root is None:
        return
    lines = context["work"].read_text(encoding="utf-8").split("\n")
    witness = context["sources"]["printed_witness"]
    system = _system(context)
    people = [n.strip() for n in names.replace(" and ", ",").split(",") if n.strip()] or ["Your name", "Your partner's name"]
    if not names.strip():
        print("(Write your names after NAMES = in the cell above and run both cells again; until then the lines say \"Your name\".)\n")
    print(f"1. The print we use. In the first <manifestation>, the Peters print, after its line\n"
          f"   <identifier type=\"URI\">… (line {_nth_line(lines, '<identifier type=\"URI\"')}), add:\n")
    print(f'        <identifier type="IMSLP">{witness["imslp_file_id"]}</identifier>')
    print(f'        <identifier type="SHA-256">{witness["sha256"]}</identifier>')
    changes = root.findall(f"{{{MEI_NS}}}meiHead/{{{MEI_NS}}}revisionDesc/{{{MEI_NS}}}change")
    number = max((int(c.get("n")) for c in changes if (c.get("n") or "").isdigit()), default=0) + 1
    persons = "\n".join(f"            <persName>{person}</persName>" for person in people)
    print(f"\n2. Your work. Before the line </revisionDesc> (line {_nth_line(lines, '</revisionDesc')}), add:\n")
    print(f"""      <change n="{number}">
        <respStmt>
{persons}
        </respStmt>
        <changeDesc>
          <p>Linked every bar to its box on pages 42–43 of the Peters print (IMSLP {witness['imslp_file_id']}) with CAMAT 0.2.8. Restored the printed marks of page {system['page']}, system {system['number']} ({_bars(system['bars'])}) from the print.</p>
        </changeDesc>
        <date isodate="{datetime.date.today().isoformat()}"/>
      </change>""")


def check_header(context):
    """Are the two header tasks done?"""
    root = _read(context["work"])
    if root is None:
        return
    m = f"{{{MEI_NS}}}"
    witness = context["sources"]["printed_witness"]
    peters = root.find(f"{m}meiHead/{m}manifestationList/{m}manifestation")
    values = [_text(i) for i in peters.findall(f"{m}identifier")]
    imslp = any(v.removeprefix("IMSLP") == str(witness["imslp_file_id"]) for v in values)
    sha = witness["sha256"] in values
    before = len(ET.parse(context["before"]).getroot().findall(f"{m}meiHead/{m}revisionDesc/{m}change"))
    changes = root.findall(f"{m}meiHead/{m}revisionDesc/{m}change")
    new = changes[before:]
    named = [c for c in new if any(_text(p) and "Your" not in _text(p) for p in c.iter(f"{m}persName"))]
    print(("✓" if imslp else "✗") + " The Peters print has the IMSLP file number 28165.")
    print(("✓" if sha else "✗") + " The Peters print has the fingerprint (SHA-256) of the scan.")
    if not new:
        print("✗ No new <change> yet.")
    elif not named:
        print("✗ The new <change> does not have your names yet.")
    else:
        print(f"✓ A new <change>, n=\"{new[-1].get('n')}\", by {_who(new[-1], {})}, dated {_date(new[-1])}.")
    if len(new) > 1:
        print(f"! {len(new)} new <change>s: one is enough for today's work.")


# --- The notes of your system --------------------------------------------------------

def _events(layer):
    """[(beat, element)] for the notes, chords and rests of a layer. A beat is an eighth (6/8),
    counted from 1 at the start of the bar: the number to write in tstamp."""
    events, position = [], Fraction(0)

    def walk(element, beam):
        nonlocal position
        if _name(element) in ("note", "chord", "rest", "space"):
            dur, dots = element.get("dur"), int(element.get("dots") or 0)
            length = Fraction(8, int(dur)) * (2 - Fraction(1, 2 ** dots)) if dur and dur.isdigit() else Fraction(0)
            events.append((1 + position, element, beam))
            position += 0 if element.get("grace") else length
            return
        for child in element:
            walk(child, element if _name(element) == "beam" else beam)
    walk(layer, None)
    return events


def _beat(value):
    whole, part = divmod(value, 1)
    return f"{int(whole)}" + ({Fraction(1, 4): "¼", Fraction(1, 2): "½", Fraction(3, 4): "¾"}.get(part, "") if part else "")


def show_notes(context):
    """The notes of your system, bar by bar and staff by staff: beat, pitch and xml:id."""
    root = _read(context["work"])
    if root is None:
        return
    bars = _system(context)["bars"]
    music = root.find(f"{{{MEI_NS}}}music")
    hands = {"1": "right hand (staff 1)", "2": "left hand (staff 2)"}
    for measure in music.iter(f"{{{MEI_NS}}}measure"):
        if measure.get("n") not in bars:
            continue
        print(f"Bar {measure.get('n')}   <measure n=\"{measure.get('n')}\" xml:id=\"{measure.get(XML_ID)}\">")
        for staff in measure.findall(f"{{{MEI_NS}}}staff"):
            print(f"  {hands.get(staff.get('n'), 'staff ' + staff.get('n'))}")
            for layer in staff.findall(f"{{{MEI_NS}}}layer"):
                events = _events(layer)
                index = 0
                while index < len(events):
                    beat, element, beam = events[index]
                    group = [element]
                    # Short beamed notes are shown on one line: their first and last are slur ends.
                    while (beam is not None and _name(element) == "note" and element.get("dur") in ("16", "32")
                           and index + 1 < len(events) and events[index + 1][2] is beam
                           and _name(events[index + 1][1]) == "note"):
                        index += 1
                        group.append(events[index][1])
                    if _name(element) == "rest":
                        print(f"    beat {_beat(beat):<4} rest")
                    elif _name(element) == "chord":
                        notes = element.findall(f"{{{MEI_NS}}}note")
                        print(f"    beat {_beat(beat):<4} {'+'.join(map(_pitch, notes)):<18} chord {element.get(XML_ID)} "
                              f"(notes {', '.join(n.get(XML_ID) for n in notes)})")
                    elif _name(element) == "note":
                        pitches = " ".join(_pitch(n) for n in group)
                        print(f"    beat {_beat(beat):<4} {pitches:<18} {' '.join(n.get(XML_ID) for n in group)}")
                    index += 1


def measure_xml(root, n):
    """The XML text of <measure n="n"> in the music, as in the file, without the MEI namespace."""
    for measure in root.find(f"{{{MEI_NS}}}music").iter(f"{{{MEI_NS}}}measure"):
        if measure.get("n") == str(n):
            excerpt = copy.deepcopy(measure)
            excerpt.tail = None
            lines = ET.tostring(excerpt, encoding="unicode").replace(f' xmlns="{MEI_NS}"', "").split("\n")
            indent = min((len(line) - len(line.lstrip()) for line in lines[1:] if line.strip()), default=0)
            return "\n".join([lines[0]] + [line[indent:] for line in lines[1:]])
    raise ValueError(f"No measure n=\"{n}\" in the music.")


def slur_prompt(context):
    """A prompt for Copilot: the slurs of your system, with its bars from the file."""
    root = _read(context["work"])
    if root is None:
        return
    system = _system(context)
    measures = "\n".join(measure_xml(root, n) for n in system["bars"])
    print(f'''I am editing an MEI file (MEI 5.1) of Grieg, Vöglein (Little bird), Op. 43 No. 4.
Add the slurs printed in system {system['number']} of page {system['page']}, {_bars(system['bars'])}:

- REPLACE THIS LINE: for each slur, the xml:id of its first and last note, its staff (1 or 2),
  and whether it curves above or below the notes.

Add one <slur> element per slur, inside the measure where it starts, after its <staff>
elements, with startid and endid pointing to the xml:id of the two notes (with a # in
front), staff, curvedir and an xml:id that begins with "seminar-slur-". Change nothing else.

Here are the measures from the file:

{measures}''')


# --- What changed -------------------------------------------------------------------------

def _key(element):
    attributes = tuple(sorted((k, v) for k, v in element.attrib.items() if k != XML_ID))
    return _name(element), element.get(XML_ID), attributes, (element.text or "").strip()


def _places(root):
    """Map every element to where it is: a bar of the music, the header, or elsewhere."""
    m = f"{{{MEI_NS}}}"
    place = {element: "the header" for element in root.find(f"{m}meiHead").iter()}
    music = root.find(f"{m}music")
    for element in music.iter():
        place[element] = "the music, outside the bars"
    for measure in music.iter(f"{m}measure"):
        for element in measure.iter():
            place[element] = measure.get("n")
    return place


def _describe(element, by_id, place):
    """One line for an element: its name, text, notes it points to, other attributes."""
    name = _name(element)
    parts = [f"<{name}>"]
    if name == "note":
        parts.append(_pitch(element))
    text = _text(element)
    if text:
        parts.append(f'"{textwrap.shorten(text, 70)}"')
    if element.get(XML_ID):
        parts.append(f"(xml:id {element.get(XML_ID)})")
    details = []
    for attribute in ("startid", "endid"):
        target = (element.get(attribute) or "").lstrip("#")
        if target:
            word = "from" if attribute == "startid" else "to"
            note = by_id.get(target)
            if note is None:
                details.append(f"{word} #{target} (no element in the file has this xml:id!)")
            else:
                where = place.get(note, "?")
                where = f"bar {where}" if where.isdigit() else where
                pitch = _pitch(note) if _name(note) == "note" else f"<{_name(note)}>"
                details.append(f"{word} {pitch} in {where}")
    for attribute, value in sorted(element.attrib.items()):
        if attribute not in (XML_ID, "startid", "endid") and "}" not in attribute:
            details.append(f'{attribute}="{value}"')
    return " ".join(parts) + (": " + ", ".join(details) if details else "")


def _changes(context):
    """[(place, sign, text, note)] for the differences between the file before editing and work.mei."""
    old, new = _read(context["before"]), _read(context["work"])
    if old is None or new is None:
        return None, []
    bars = _system(context)["bars"] if "system" in context else None
    old_elements, new_elements = list(old.iter()), list(new.iter())
    old_place, new_place = _places(old), _places(new)
    old_ids = {e.get(XML_ID): e for e in old_elements if e.get(XML_ID)}
    new_ids, duplicates = {}, set()
    for element in new_elements:
        if element.get(XML_ID):
            if element.get(XML_ID) in new_ids:
                duplicates.add(element.get(XML_ID))
            new_ids[element.get(XML_ID)] = element

    def note(element, place, asked):
        if not asked:
            return "not part of the tasks"
        if place.isdigit() and bars and place not in bars:
            return "outside your system"
        return ""

    changes = []
    matcher = difflib.SequenceMatcher(None, [_key(e) for e in old_elements], [_key(e) for e in new_elements],
                                      autojunk=False)
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            continue
        removed, added = old_elements[i1:i2], new_elements[j1:j2]
        same_id = {e.get(XML_ID) for e in removed if e.get(XML_ID)} & {e.get(XML_ID) for e in added}
        for element in added:
            place = new_place.get(element, "?")
            if element.get(XML_ID) in same_id:
                was = next(e for e in removed if e.get(XML_ID) == element.get(XML_ID))
                diffs = [f'{a}: "{was.get(a)}" → "{element.get(a)}"' for a in sorted(set(was.attrib) | set(element.attrib))
                         if was.get(a) != element.get(a)]
                if (was.text or "").strip() != (element.text or "").strip():
                    diffs.append(f'text: "{(was.text or "").strip()}" → "{(element.text or "").strip()}"')
                label = f"<{_name(element)}> {element.get(XML_ID)}"
                if _name(element) == "note" and _pitch(was) != _pitch(element):
                    label += f" ({_pitch(was)} → {_pitch(element)})"
                asked = _name(element) == "facsimile" and set(element.attrib) ^ set(was.attrib) <= {"decls"}
                changes.append((place, "~", f"changed {label}: " + "; ".join(diffs), note(element, place, asked)))
            elif not any(element in parent for parent in added if parent is not element):
                asked = (_name(element) in MARKS | HEADER_TASKS and element.get(XML_ID) not in old_ids
                         and (place == "the header") == (_name(element) in HEADER_TASKS))
                changes.append((place, "+", "new " + _describe(element, new_ids, new_place), note(element, place, asked)))
        for element in removed:
            if element.get(XML_ID) in same_id or any(element in parent for parent in removed if parent is not element):
                continue
            place = old_place.get(element, "?")
            changes.append((place, "-", "removed " + _describe(element, old_ids, old_place), "not part of the tasks"))
    problems = [f"xml:id {i} is used more than once" for i in sorted(duplicates)]
    for element in new_elements:
        for attribute in ("startid", "endid", "plist", "facs"):
            for token in (element.get(attribute) or "").split():
                if token.startswith("#") and token[1:] not in new_ids:
                    problems.append(f"<{_name(element)}> {attribute} points to {token}, but no element has that xml:id")
    return problems, changes


def show_changes(context):
    """What changed in work.mei since before editing, in plain words. Never stops the notebook."""
    problems, changes = _changes(context)
    if problems is None:
        return
    work = context["work"].name
    if not changes and not problems:
        print(f"No changes yet: {work} is the same as before editing (spaces and line breaks are not counted).")
        return
    print(f"Changes in {work} since before editing (spaces and line breaks are not counted):")
    places = []
    for place, *_ in changes:
        if place not in places:
            places.append(place)
    for place in places:
        print(f"\n{'Bar ' + place if place.isdigit() else place[0].upper() + place[1:]}")
        for _, sign, text, flag in (c for c in changes if c[0] == place):
            print(f"  {sign} {text}" + (f"   ← {flag}" if flag else ""))
    flagged = [c for c in changes if c[3]]
    if flagged:
        print(f"\n{len(flagged)} change{'s' if len(flagged) != 1 else ''} marked ←: look at each one in the print. "
              "Keep it if the print shows it and it belongs to your task; otherwise undo it in the editor.")
    if problems:
        print("\nTo fix:")
        for problem in problems:
            print("  ✗", problem)


# --- Check and review -----------------------------------------------------------------------

def check_file(context):
    """CAMAT's MEI checker on work.mei: what is new, and what is gone, since before editing."""
    if _read(context["work"]) is None:
        return
    project = context["project"]
    now = _validate(context["work"], project / "edition/validation-editing")
    then = _validate(context["before"], project / "edition/validation-before-editing")
    if now is None or then is None:
        print("The checker did not run. Run the cell again; if it still fails, ask your teacher.")
        return

    def count(findings):
        counts = {}
        for finding in findings:
            counts.setdefault(_finding_key(finding), []).append(finding)
        return counts
    old, current = count(then), count(now)
    new = [f for key, found in current.items() for f in found[len(old.get(key, [])):]]
    gone = [f for key, found in old.items() for f in found[len(current.get(key, [])):]]
    print(f"Checked {context['work'].name} against the MEI rules (schema), its links and the page images, and with Verovio.")
    if not new:
        print(f"✓ No new problems. (The file before editing had {len(then)} remarks; they are not shown.)")
    for finding in new:
        where = f"bar {finding['measure_n']}" if finding.get("measure_n") else "outside the bars"
        print(f"  {'✗' if finding['severity'] == 'error' else '!'} line {finding.get('line') or '?'} ({where}): {finding['message']}")
    if any(f["severity"] == "error" for f in new):
        print("Fix the lines marked ✗, save, and run this cell again.")
    if gone:
        print(f"\nGone since before editing ({len(gone)}):")
        for finding in gone:
            print(f"  ✓ {finding['message']}")
    print(f"The full report is in {_short(project / 'edition/validation-editing', project)}.")


def show_review(context, width=900):
    """For the reviewers: the new marks bar by bar, then the system in the print
    (and any other bar with changes, on its own)."""
    problems, changes = _changes(context)
    if problems is None:
        return
    bars = sorted({c[0] for c in changes if c[0].isdigit()}, key=int)
    others = [c for c in changes if not c[0].isdigit()]
    if not bars and not others:
        print("No changes to review yet.")
        return
    system = _system(context)
    for bar in bars:
        print(f"Bar {bar}")
        for _, sign, text, flag in (c for c in changes if c[0] == bar):
            print(f"  {sign} {text}" + (f"   ← {flag}" if flag else ""))
    if others:
        print(f"Elsewhere: {len(others)} change{'s' if len(others) != 1 else ''} (the header and so on), "
              "listed by show_changes.")
    if problems:
        print("To fix:", *problems, sep="\n  ✗ ")
    print(f"\nPage {system['page']}, system {system['number']}, in the print:")
    show_system(context)
    boxes = _boxes(context)
    for bar in (b for b in bars if b not in system["bars"]):
        print(f"Bar {bar}, outside the system, in the print:")
        _crop(*boxes[bar], width=width)


REVIEW_DECISIONS = ("accept", "correct", "unresolved")


def _sheet_entry(sign, text, flag):
    return [f"- `{sign} {text}`" + (f" ← {flag}" if flag else ""), "    - Decision:", "    - Why:"]


def _read_sheet(text):
    """{change: what the reviewers wrote after Decision:}, for the entries of a review sheet."""
    entries, current = {}, None
    for line in text.split("\n"):
        if line.startswith("## Unresolved"):
            break
        found = re.match(r"- `(.+)`", line)
        if found:
            current = found.group(1)
            entries[current] = ""
        elif current is not None and line.strip().lstrip("-* ").lower().startswith("decision:"):
            entries[current] = line.split(":", 1)[1].strip()
            current = None
    return entries


def review_sheet(context, names=""):
    """The peer-review sheet edition/reviews/page-P-system-S.md: one entry per change in the bars.
    Written once and then kept; run again, it counts the decisions and lists the changes not on it.
    One file per system, so that the pairs' sheets do not collide when session 4 combines them."""
    problems, changes = _changes(context)
    if problems is None:
        return
    project, system = context["project"], _system(context)
    sheet = project / f"edition/reviews/page-{system['page']}-system-{system['number']}.md"
    entries = sorted((c for c in changes if c[0].isdigit()), key=lambda c: int(c[0]))
    if problems:
        print("To fix first:", *problems, sep="\n  ✗ ")
        print()
    if not sheet.is_file():
        if not entries:
            print("No changes in the bars yet, so there is nothing to review.")
            return
        lines = [f"# Peer review: page {system['page']}, system {system['number']} ({_bars(system['bars'])})", "",
                 f"Restored by: {names.strip()}  ", "Reviewed by:  ",
                 f"Sheet written: {datetime.date.today().isoformat()}", "",
                 "Compare each change with the print. After *Decision* write accept, correct or unresolved, "
                 "and after *Why* a few words, for example: the print has the slur below the staff. "
                 "Correct means that the restoring pair fixes it in work.mei while you watch. "
                 "Look twice at the changes marked ←."]
        for bar in dict.fromkeys(c[0] for c in entries):
            lines += ["", f"## Bar {bar}", ""]
            for _, sign, text, flag in (c for c in entries if c[0] == bar):
                lines += _sheet_entry(sign, text, flag)
        lines += ["", "## Unresolved: to discuss in class", "", "- ", ""]
        sheet.parent.mkdir(exist_ok=True)
        sheet.write_text("\n".join(lines), encoding="utf-8")
        print(f"Wrote the review sheet, with {len(entries)} entr{'ies' if len(entries) != 1 else 'y'}: {_short(sheet, project)}")
        print("Open it in the editor next to the print and fill it in. Run this cell again when you have finished.")
        return

    listed = _read_sheet(sheet.read_text(encoding="utf-8"))
    current = {f"{sign} {text}": (bar, flag) for bar, sign, text, flag in entries}
    print(f"The review sheet is already there and is kept as it is: {_short(sheet, project)}")

    def word(value):
        return value.split()[0].lower().strip(".,;:!") if value.split() else ""
    decided = [change for change, value in listed.items() if word(value) in REVIEW_DECISIONS]
    unclear = [change for change, value in listed.items() if value and word(value) not in REVIEW_DECISIONS]
    empty = [change for change, value in listed.items() if not value]
    counts = ", ".join(f"{n} {w}" for w in REVIEW_DECISIONS
                       if (n := sum(word(listed[c]) == w for c in decided)))
    done = listed and not unclear and not empty
    print(f"{'✓' if done else '!'} {len(decided)} of {len(listed)} entries have a decision" + (f": {counts}." if counts else "."))
    for change in empty:
        print(f"  no decision yet: {textwrap.shorten(change, 100)}")
    for change in unclear:
        print(f"  write accept, correct or unresolved: {textwrap.shorten(change, 90)}")
    missing = [change for change in current if change not in listed]
    gone = [change for change in listed if change not in current]
    if missing:
        print(f"\nIn work.mei, but not on the sheet ({len(missing)}): a mark added or corrected after the sheet was written. "
              "Copy each entry under its bar on the sheet, and review it:")
        for change in missing:
            bar, flag = current[change]
            print(f"\nBar {bar}", *_sheet_entry(*change.split(" ", 1), flag), sep="\n")
    if gone:
        print(f"\nOn the sheet, but no longer in work.mei ({len(gone)}): corrected or undone since. "
              "Write under Why what was done:")
        for change in gone:
            print(f"  {textwrap.shorten(change, 100)}")

In [ ]:
PROJECT = Path.cwd() / "grieg_edition_work"
GRIEG = open_edition(PROJECT)
WORK_MEI = GRIEG["work"]

## 1. What the header is for

Remember the Schumann piece of session 2: in bars 1 and 5, the encoding has an A3 in the left hand where our print has an F3. Nobody typed it wrong. The encoding was made from another source, a file from the KernScores collection, not from the Clara Schumann edition we compared it with. Its header said so, for anyone who read it.

Reusing an encoding saves a lot of typing. The header tells you what you are reusing: from which source, by whom, and with which changes since. The next cell shows the header of `work.mei` in plain words, with the lines where you find each part in the file.

In [ ]:
show_header(GRIEG)

Read the summary, and look up each part in `work.mei` with the line numbers:

- The header names two sources. Which one is our print? Compare its plate number with the number at the bottom of page 42.
- Read the first two changes. From which file were the notes made, and when? Does the header say that anyone compared them with the Peters print?
- Changes 3 and 5 point to two tools, `#xsl_ppq` and `#xsl_header`, that the file does not describe. A `#` link works like `facs="#zone-…"` in block 1: it must point to an `xml:id` in the same file. These two point to nothing, and CAMAT's checker reports them as errors.

## 2. Your names and your system

Each pair restores the marks of one system. Write your names and the system your teacher gave you, then run the cell. The names go into the header of `work.mei`, and so into your public repository: names, initials or GitHub usernames are all fine.

In [ ]:
NAMES = ""   # for example "Anna and Ben"
PAGE = 42    # 42 or 43
SYSTEM = 1   # 1 to 5, from the top of the page

choose_system(GRIEG, PAGE, SYSTEM)

## 3. Add to the header

The header does not know two things yet: exactly which scan we use, and what you did. The next cell prints the lines to add, with your names, your system and today's date, and the line numbers where they go.

1. Open `work.mei` in the editor, go to the line the cell names (Ctrl+G, then the line number), and paste the two `<identifier>` lines.
2. Paste the `<change>` before `</revisionDesc>`. Read it: does it say what you did? Change the words if not.
3. Save the file, then run the cell after next.

In [ ]:
header_lines(GRIEG, NAMES)

In [ ]:
check_header(GRIEG)

When the check shows three ✓, record your change: in VS Code's **Source Control** view of the edition folder, write the message *Add the scan and our work to the header*, click **Commit**, then **Sync Changes**. Do the same after each task below, with a message that says what you did.

<details><summary>Why the fingerprint of the scan?</summary>

IMSLP sometimes replaces a scan with a better one under the same file number. Our boxes would then no longer fit. With the fingerprint (SHA-256), anyone can check later that they have exactly the file we linked to: block 1 did the same check after the download.

</details>

**Extension: repair the two broken links.** In `<appInfo>` (in `<encodingDesc>`), after the last `</application>`, add a description of each tool. Then the check in part 5 lists the two errors as gone.

```xml
<application xml:id="xsl_ppq">
  <name>ppq.xsl</name>
</application>
<application xml:id="xsl_header">
  <name>Header.xsl</name>
</application>
```

## 4. Restore the marks of your system

### 4a. Your system in the print and in the score

The next cell shows your system in the print, with its bar numbers. The cell after it starts the live score from block 1: click a bar to see its box on the page. Then open `work.mei` in the editor next to the notebook.

In [ ]:
show_system(GRIEG)

In [ ]:
from camat import launch_interactive_facsimile_viewer

if "viewer" in globals():
    viewer.stop_watch()
viewer = launch_interactive_facsimile_viewer(
    str(WORK_MEI), auto_watch_mei=True, verovio_breaks="encoded",
    verovio_options={"header": "none", "footer": "none"},
)

Compare the print with the score. The encoding has the notes, the ties, the staccato dots and the accents. Of the marks below and between the staves it has only two: *p* in bar 1 and *pp* in bar 10. Slurs, pedal marks, fingerings, the tempo words and most dynamics are missing. You restore them for your system.

### 4b. The notes of your system

A mark in MEI says where it belongs: in which bar, on which staff, and on which *beat*. The beat is the number you write in `tstamp`. In 6/8 a bar has six beats, one for each eighth note; the short 32nd notes fall between them (¼, ½, ¾). A slur instead names its first and last note by their `xml:id`.

The next cell lists the notes of your system: for each bar and staff, the beat, the pitch and the `xml:id`. Short notes beamed together are on one line.

In [ ]:
show_notes(GRIEG)

### 4c. By hand: the dynamics

Look for the dynamics of your system in the print: letters such as *p*, *pp*, *f* or *fz*, words such as *cresc.*, and *hairpins*, the long opening or closing wedges. Add each one inside the `<measure>` where it begins, after its two `<staff>` elements, before `</measure>`. Change the beat, the staff and the letters to what the print shows:

```xml
<dynam xml:id="seminar-dynam-1" staff="1" place="below" tstamp="1">pp</dynam>
<hairpin xml:id="seminar-hairpin-1" form="dim" staff="1" place="below" tstamp="2" tstamp2="0m+6"/>
```

- `staff="1"` is the upper staff (right hand), `staff="2"` the lower one; `place` puts the mark below or above that staff.
- A hairpin has a beginning and an end: `form="cres"` opens, `form="dim"` closes; `tstamp2="0m+6"` means "in this bar (0 bars later), on beat 6", `"1m+2"` would be beat 2 of the next bar.
- Every new element needs its own `xml:id`: count on, `seminar-dynam-2`, `seminar-hairpin-2` and so on.
- **System 1 of page 42** also restores the tempo words: `<tempo xml:id="seminar-tempo-1" staff="1" place="above" tstamp="1">Allegro leggiero.</tempo>` in bar 1.

Save, look at the score, and run the next cell. It lists what changed since before editing.

In [ ]:
show_changes(GRIEG)

Changes marked **not part of the tasks** or **outside your system** need a second look: keep them only if the print shows them and they belong to your task. Then commit and sync, for example with the message *Add the dynamics of system 3*.

### 4d. With Copilot: the slurs

The slurs are the long curved lines over or under groups of notes. For a slur, the model needs to know its first and last note. The next cell writes a prompt with the bars of your system from your file.

1. Copy the prompt into Copilot's chat (View → Chat).
2. Replace the line that begins with `REPLACE THIS LINE` with one line per slur, for example: *from d418889e30 to d418889e221, staff 1, above*. Take the `xml:id`s from the list in 4b.
3. Send it, look at the change Copilot proposes, accept it, and save `work.mei`.
4. Look at the score: does each slur begin and end where the print has it?

**Without Copilot** (in the cloud, or if Copilot does not answer), write each slur by hand, in the `<measure>` where it begins, after its `<staff>` elements. `startid` and `endid` name its first and last note, with a `#` in front of the `xml:id`:

```xml
<slur xml:id="seminar-slur-1" staff="1" curvedir="above" startid="#d418889e30" endid="#d418889e221"/>
```

In [ ]:
slur_prompt(GRIEG)

In [ ]:
show_changes(GRIEG)

Read each new `<slur>` line: it names its two notes, with pitch and bar. If the model chose other notes, correct `startid` or `endid` by hand. Language models often change more than you asked for: check every line marked with ←. Then commit and sync.

### 4e. By hand: the pedal marks

Under the lower staff, *Ped.* means "press the right pedal" and ✻ "release it". In MEI both are a `<pedal>` element, with `dir="down"` for *Ped.* and `dir="up"` for ✻:

```xml
<pedal xml:id="seminar-pedal-1" staff="2" tstamp="1" dir="down"/>
<pedal xml:id="seminar-pedal-2" staff="2" tstamp="4" dir="up"/>
```

Give each mark the beat under which it is printed. If the ✻ is in the next bar, its `<pedal>` goes into that bar.

In [ ]:
show_changes(GRIEG)

Then commit and sync.

**Keep the rest as it is.** Notes, rhythms, `xml:id`s and links stay unchanged. A mark that the print has and the encoding left out is restored as an ordinary part of the transcription.

**Extensions**

- **Fingerings,** the small numbers over the notes: `<fing xml:id="seminar-fing-1" staff="1" tstamp="1" place="above">2</fing>`.
- **The words *poco ritardando*** (page 43, system 5): `<dir xml:id="seminar-dir-1" staff="1" place="below" tstamp="…" tstamp2="…">poco ritardando</dir>`.
- **The 8va line at the page turn.** The `<octave>` in bar 20 has `tstamp2="4"`: it ends on beat 4 of bar 20. Where does the dotted line end in the print? Write that place as `tstamp2`, for example `"1m+1"` for beat 1 of the next bar.

## 5. Check the file

The score looks right, but is the file still correct MEI, with all its links? The next cell checks `work.mei` against the MEI rules (the *schema*), follows the links, and draws the file with Verovio. It shows only what is new since before editing, and what is gone.

In [ ]:
check_file(GRIEG)

## 6. Peer review

Swap with another pair: you review their system on their laptop, they review yours. The next cell shows each bar with new marks: the marks in words, and the bar in the print. On GitHub, their commits show the same changes line by line. For every mark, check against the print:

- the right bar and the right beat;
- for a slur, the right first and last note;
- the right staff, above or below;
- nothing changed that should not be (the lines marked ←).

A file that passes the check in part 5 is a valid file, not yet a checked transcription: only the comparison with the print shows that.

In [ ]:
show_review(GRIEG)

Write down what you decide. The next cell writes the review sheet, with one entry for each change in the bars, into the folder `grieg_edition_work/edition/reviews/`, named after the page and the system. Open it in the editor and write your names after *Reviewed by*. Under each entry, write your decision:

- **accept**: the print shows the mark, and it is in the right place;
- **correct**: something is wrong; the pair that restored the mark fixes it in `work.mei` while you watch;
- **unresolved**: you could not decide together. Say why, and write it under *Unresolved* at the end of the sheet.

Save the sheet and run the cell again: it counts the decisions, and lists any change that is not on the sheet yet, such as a corrected mark. The sheet stays next to `work.mei` in your repository: when it is complete, commit and sync it (*Add the review sheet*), together with any corrections. Your names on it are public too: names, initials or GitHub usernames.

In [ ]:
review_sheet(GRIEG, NAMES)

**Checkpoint:** every entry on the sheet has a decision.

## At the end

Your work is in `work.mei` and in the commits of your repository. Check in VS Code's Source Control view that nothing is left to commit or sync. The last notebook, `03_grieg_live_page`, turns the repository into a live page. Run the last cell to stop the score from watching the file.

In [ ]:
viewer.stop_watch()